# ICT-15c — Méta-proxy d'obstruction : structure des désaccords entre proxys

**Issue** : [#7395](https://github.com/jsboige/CoursIA/issues/7395)  
**Epic** : [#4588](https://github.com/jsboige/CoursIA/issues/4588) (IIT -> ICT)

## Cadrage

L'intuition directrice (lecture cohomologie-obstruction, cf. doc Grothendieck
mergée le 19/07/2026) est que **l'obstruction elle-même est informative** :
quand plusieurs proxys de complexité/intégration (spectral, sensitivity,
free-energy) ne « recollent » pas en une mesure globale unique, le *motif*
de leur désaccord porte de l'information sur le substrat.

## Acceptance (issue #7395, falsifiable)

- **Positif** : motif de désaccord **stable cross-substrat** (montrer qu'il
  n'est pas arbitraire — même structure de désaccord sur ≥ 2 substrats).
- **Négatif honnête** : motif = bruit (pas de structure stable) → verdict
  négatif documenté, la jambe meurt proprement.
- **Premier livrable concret** : cas où la lecture-obstruction change une
  décision expérimentale ou une visualisation (pas un habillage rétrospectif).

## Discipline (HARD)

**Ne pas nommer l'objet** tant qu'il n'est pas stabilisé. Le nom
(grothendieckien, « structure » et non « mesure ») vient *après* la
stabilisation empirique, pas avant. Pas de lettre grecque décorée sur un
objet non encore constaté.

## Substrats pilotes (3 instrumentés)

1. **Gray-Scott** (réaction-diffusion, Turing patterns) — `ict.reaction_diffusion.GrayScott`
2. **Axelrod** (morphodynamique stratégique, fin de cycle réplicateur) — `ict.strategic_morphodynamics`
3. **Grokking** (compression crossover) — simulé ici via marche aléatoire biaisée
   sur alphabet à 4 états (proxy minimal).

## Livrable de cette notebook

Application du module `ict.meta_proxy` sur 3 substrats synthétiques,
avec verdict falsifiable `STABLE / NOISE / INCONCLUSIVE` à la fin.

In [1]:
# Imports et substrats pilotes
import sys
from pathlib import Path

ICT_ROOT = Path('.').resolve()
sys.path.insert(0, str(ICT_ROOT))

import numpy as np
import matplotlib
matplotlib.use('Agg')  # rendu PNG en batch
import matplotlib.pyplot as plt

from ict import spectral as SP
from ict import sensitivity as SE
from ict import reaction_diffusion as RD
from ict import strategic_morphodynamics as SM
from ict.meta_proxy import (
    proxy_signature,
    obstruction_vector,
    cross_substrat_obstruction,
)

np.random.seed(20260720)  # determinisme cross-execution
print("meta_proxy loaded. Substrats pilotes : Gray-Scott, Axelrod, Grokking.")

meta_proxy loaded. Substrats pilotes : Gray-Scott, Axelrod, Grokking.


In [2]:
# --- Substrat 1 : Gray-Scott Turing patterns ---
# API reelle de ict.reaction_diffusion.GrayScott :
#   - constructeur prend F, k, Du, Dv, dt (PAS de `size`)
#   - seed(n=64, rng=...) -> (U, V) initiaux
#   - run(U, V, steps) -> (U, V, snapshots)
# On integre 800 pas avec F/k dans la fenetre Turing pattern (F=0.035, k=0.065).
gs = RD.GrayScott(F=0.035, k=0.065, Du=0.16, Dv=0.08, dt=1.0)
seed_rng = np.random.default_rng(20260720)
U_init, V_init = gs.seed(n=64, rng=seed_rng)
U_final, V_final, _ = gs.run(U_init, V_init, steps=800)

# Binarisation : pixels V > 0.5 en 'pattern' (1), sinon 'background' (0)
v = V_final
binary_grid = (v > 0.5).astype(int)
n_pattern = int(binary_grid.sum())
n_bg = int((1 - binary_grid).sum())
ratio = n_pattern / (n_pattern + n_bg)
print(f"Gray-Scott : pattern={n_pattern}, background={n_bg}, ratio={ratio:.3f}")

# Trajectoire discrete : on scan les pixels ligne par ligne -> sequence de 0/1
gray_scott_states = binary_grid.flatten().tolist()
n_symbols_gs = 2
print(f"Longueur trajectoire Gray-Scott : {len(gray_scott_states)} pixels")

Gray-Scott : pattern=0, background=4096, ratio=0.000
Longueur trajectoire Gray-Scott : 4096 pixels


In [3]:
# --- Substrat 2 : Axelrod end-of-cycle replicator ---
# API reelle de ict.strategic_morphodynamics :
#   - make_strategies(rng) -> Dict[str, Strategy]
#   - payoff_matrix(strategies, n_rounds, n_reps, rng) -> A  (matrice de gain)
#   - replicator_trajectory(A, x0, n_steps) -> traj (n_steps+1, n_strategies)
# On calibre la matrice de gain puis on integre la dynamique.
rng = np.random.default_rng(20260720)
strategies = SM.make_strategies(rng)
A = SM.payoff_matrix(strategies, n_rounds=200, n_reps=3, rng=rng)
n_strat = A.shape[0]
# Frequences initiales uniformes (chaque strategie a 1/n_strat).
x0 = np.full(n_strat, 1.0 / n_strat)
traj = SM.replicator_trajectory(A, x0, n_steps=400)

# Trajectoire discrete : on binarise la stabilite de la dominance.
# Pour chaque pas, on prend l'index de la strategie dominante, puis on
# enregistre 1 si elle reste dominante au pas suivant, 0 sinon.
dom_idx = np.argmax(traj, axis=1)
axelrod_states = [int(dom_idx[i] == dom_idx[i - 1]) for i in range(1, len(dom_idx))]
n_symbols_ax = 2
print(f"Axelrod : trajectoire de {len(axelrod_states)} pas (stabilité de la dominance).")

Axelrod : trajectoire de 400 pas (stabilité de la dominance).


In [4]:
# --- Substrat 3 : Grokking compression crossover (proxy minimal) ---
# Marche aléatoire biaisée : phase 1 = haute entropie (alphabet 4 etats),
# phase 2 = compression vers un etat attracteur (un seul etat visité).
# Le crossover est placé au pas 200 (milieu de trajectoire).
n_steps_g = 400
states_g = []
for t in range(n_steps_g):
    if t < 200:
        # Phase haute entropie : equiprobable parmi 4 etats
        s = int(rng.integers(0, 4))
    else:
        # Phase compression : 90% du temps sur l'etat 0, 10% sur les autres
        if rng.random() < 0.90:
            s = 0
        else:
            s = int(rng.integers(1, 4))
    states_g.append(s)
grokking_states = states_g
n_symbols_gk = 4
print(f"Grokking : trajectoire de {len(grokking_states)} pas, crossover au pas 200.")

Grokking : trajectoire de 400 pas, crossover au pas 200.


In [5]:
# --- Calcul des signatures multi-proxys ---
# On enrobe les fonctions ict.spectral / ict.sensitivity pour matcher
# le prototype (states, n_symbols) -> float attendu par meta_proxy.
def spec_gap_wrap(states, n_symbols):
    return float(SP.spectral_summary(states, n_symbols)['spectral_gap'])

def sens_mean_wrap(states, n_symbols):
    # f(x) = x % 2 : fonction bivalente simple
    f = lambda x: x % 2
    return float(SE.sensitivity_distribution(states, n_symbols, f)['mean'])

def sens_max_wrap(states, n_symbols):
    f = lambda x: x % 2
    return float(SE.sensitivity_distribution(states, n_symbols, f)['max'])

sig_gs = proxy_signature(
    gray_scott_states, n_symbols_gs,
    spectral_fn=spec_gap_wrap,
    sensitivity_mean_fn=sens_mean_wrap,
    sensitivity_max_fn=sens_max_wrap,
)
sig_ax = proxy_signature(
    axelrod_states, n_symbols_ax,
    spectral_fn=spec_gap_wrap,
    sensitivity_mean_fn=sens_mean_wrap,
    sensitivity_max_fn=sens_max_wrap,
)
sig_gk = proxy_signature(
    grokking_states, n_symbols_gk,
    spectral_fn=spec_gap_wrap,
    sensitivity_mean_fn=sens_mean_wrap,
    sensitivity_max_fn=sens_max_wrap,
)

print("=== Signatures multi-proxys ===")
for nom, sig in [("gray_scott", sig_gs), ("axelrod", sig_ax), ("grokking", sig_gk)]:
    print(f"{nom:>12}: spectral_gap={sig['spectral_gap']:.4f}, "
          f"sens_mean={sig['sensitivity_mean']:.4f}, "
          f"sens_max={sig['sensitivity_max']:.4f}")

=== Signatures multi-proxys ===
  gray_scott: spectral_gap=0.5000, sens_mean=1.0000, sens_max=1.0000
     axelrod: spectral_gap=1.0000, sens_mean=1.0000, sens_max=1.0000
    grokking: spectral_gap=0.7766, sens_mean=2.0000, sens_max=2.0000


In [6]:
# --- Vecteurs d'obstruction pairwise + aggregation cross-substrat ---
substrats = {"gray_scott": sig_gs, "axelrod": sig_ax, "grokking": sig_gk}
verdict = cross_substrat_obstruction(substrats)

print("=== Vecteurs d'obstruction (a - b normalise) ===")
noms = list(substrats.keys())
for i in range(len(noms)):
    for j in range(i + 1, len(noms)):
        vec = obstruction_vector(substrats[noms[i]], substrats[noms[j]])
        print(f"  {noms[i]:>10} vs {noms[j]:<10}: "
              f"spectral={vec['spectral_gap']:+.3f}, "
              f"sens_mean={vec['sensitivity_mean']:+.3f}, "
              f"sens_max={vec['sensitivity_max']:+.3f}, "
              f"||v||₂={vec['norm_l2']:.3f}")

print(f"\n=== Verdict cross-substrat ===")
print(f"  n_substrats = {verdict['n_substrats']}")
print(f"  pairwise_norms = {[round(x, 4) for x in verdict['pairwise_norms']]}")
print(f"  mean_norm_l2  = {verdict['mean_norm_l2']:.4f}")
print(f"  max_norm_l2   = {verdict['max_norm_l2']:.4f}")
print(f"  STABLE_threshold = {verdict['stable_threshold']}, "
      f"NOISE_threshold = {verdict['noise_threshold']}")
print(f"  VERDICT = {verdict['verdict']}")

=== Vecteurs d'obstruction (a - b normalise) ===
  gray_scott vs axelrod   : spectral=-0.333, sens_mean=+0.000, sens_max=+0.000, ||v||₂=0.333
  gray_scott vs grokking  : spectral=-0.217, sens_mean=-0.333, sens_max=-0.333, ||v||₂=0.519
     axelrod vs grokking  : spectral=+0.126, sens_mean=-0.333, sens_max=-0.333, ||v||₂=0.488

=== Verdict cross-substrat ===
  n_substrats = 3
  pairwise_norms = [0.3333, 0.5188, 0.4879]
  mean_norm_l2  = 0.4467
  max_norm_l2   = 0.5188
  STABLE_threshold = 0.05, NOISE_threshold = 0.3
  VERDICT = NOISE


In [7]:
# --- Visualisation : heatmap des pairwise norms ---
n = len(noms)
M = np.zeros((n, n))
k = 0
for i in range(n):
    for j in range(i + 1, n):
        M[i, j] = verdict['pairwise_norms'][k]
        M[j, i] = M[i, j]
        k += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(M, cmap='viridis', vmin=0, vmax=max(verdict['noise_threshold'], 0.5))
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(noms, rotation=20, ha='right')
ax.set_yticklabels(noms)
for i in range(n):
    for j in range(n):
        if i != j:
            ax.text(j, i, f"{M[i, j]:.3f}", ha='center', va='center',
                    color='white' if M[i, j] < 0.3 else 'black', fontsize=9)
ax.set_title(f"Obstruction pairwise ||v||₂ — verdict : {verdict['verdict']}")
plt.colorbar(im, ax=ax, label='||v||₂ norm')
plt.tight_layout()
plt.savefig('ICT-15c-obstruction-heatmap.png', dpi=110, bbox_inches='tight')
plt.close()
print("Heatmap sauvegardee : ICT-15c-obstruction-heatmap.png")

Heatmap sauvegardee : ICT-15c-obstruction-heatmap.png

## Interprétation

**Substrats testés** : Gray-Scott (Turing patterns, binarisation des pixels),
Axelrod (réplicateur stratégique, stabilité de la dominance), Grokking
(marche aléatoire biaisée, compression crossover à mi-parcours).

**Proxys calculés** : `spectral_gap` (gap spectral du Laplacien du graphe
de transition), `sensitivity_mean` et `sensitivity_max` (sur fonction
bivalente `f(x) = x % 2`, états visités).

**Vecteur d'obstruction** : `(a - b) / (|a| + |b| + eps)` — invariant à
l'échelle globale, antisymétrique (`obstruction(a, b) = -obstruction(b, a)`).

**Verdict falsifiable** :
- `STABLE` : motif de désaccord cohérent (mean_norm_l₂ ≤ 0.05) sur ≥ 2
  substrats — preuve empirique de l'objet.
- `NOISE` : motif aléatoire (mean_norm_l₂ ≥ 0.30) — verdict négatif honnête,
  la jambe meurt proprement.
- `INCONCLUSIVE` : entre les deux seuils — recalibrer ou augmenter
  l'échantillon.

## Suite logique (issue #7395)

Premier livrable concret = au moins un cas où la lecture-obstruction
**change une décision expérimentale ou une visualisation** (issue #7395,
acceptance positive). Si le verdict est `STABLE` ici : à documenter ; si
`NOISE` : à documenter aussi (verdict honnête). Si `INCONCLUSIVE` : à
recalibrer les seuils ou augmenter la taille d'échantillon.

## Discipline de nommage (HARD)

L'objet n'est **pas encore nommé**. Le vocabulaire utilisé reste
descriptif (`obstruction vector`, `pairwise norm`, `STABLE / NOISE`).
Aucune lettre grecque décorée sur un objet non encore constaté — la
stabilisation empirique précède le baptême.